<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/06-numpy/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6: Solutions

Try the exercises first. The notes after each solution walk through the shapes at each step, and explain the NumPy traps that each exercise is designed around.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import copy
import math
import inspect
import time

import numpy as np


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

_rng = np.random.default_rng(1234)


def check_standardize(f):
    X = np.array([[1, 10], [3, 30], [5, 50]])
    r = 1.0 / np.sqrt(2 / 3)

    def unchanged():
        X2 = np.array([[1.0, 2.0], [3.0, 5.0]])
        f(X2)
        return X2

    _scenarios("standardize", [
        ("basic", lambda: f(X), np.array([[-r, -r], [0.0, 0.0], [r, r]]),
         "subtract the column means and divide by the column stds (axis=0)"),
        ("result has float dtype", lambda: f(X).dtype.kind, "f", None),
        ("constant column becomes 0", lambda: f(np.array([[1.0, 7.0], [3.0, 7.0]])), np.array([[-1.0, 0.0], [1.0, 0.0]]),
         "where the std is 0, divide by 1 instead (np.where)"),
        ("input is not modified", unchanged, np.array([[1.0, 2.0], [3.0, 5.0]]), "don't use -= or /= on X itself"),
        ("each column ends up with mean 0 and std 1", lambda: (lambda Z: (Z.mean(axis=0), Z.std(axis=0)))(f(_rng.normal(5, 3, size=(200, 4)))),
         (np.zeros(4), np.ones(4)), None),
    ])


def check_one_hot(f):
    _scenarios("one_hot", [
        ("basic", lambda: f(np.array([0, 2, 1]), 3), np.array([[1.0, 0, 0], [0, 0, 1], [0, 1, 0]]), None),
        ("float dtype", lambda: f(np.array([1]), 2).dtype.kind, "f", None),
        ("no labels", lambda: f(np.array([], dtype=int), 3).shape, (0, 3), None),
        ("label too large", lambda: f(np.array([0, 3]), 3), Raises(ValueError), None),
        ("negative label", lambda: f(np.array([0, -1]), 3), Raises(ValueError),
         "np.eye(3)[-1] silently picks the last row. Check for negatives explicitly"),
    ])


def check_softmax(f):
    def unchanged():
        z = np.array([1.0, 2.0, 3.0])
        f(z)
        return z

    _scenarios("softmax", [
        ("1D", lambda: f(np.array([1.0, 2.0, 3.0])), np.array([0.09003057, 0.24472847, 0.66524096]), None),
        ("2D: one distribution per row", lambda: f(np.array([[0.0, 0.0], [0.0, np.log(3.0)]])), np.array([[0.5, 0.5], [0.25, 0.75]]),
         "work along the last axis, with keepdims=True"),
        ("rows sum to 1", lambda: f(_rng.normal(size=(5, 7))).sum(axis=1), np.ones(5), None),
        ("large values don't overflow", lambda: f(np.array([1000.0, 1001.0])), np.array([0.26894142, 0.73105858]),
         "subtract the max of each row before calling np.exp"),
        ("input is not modified", unchanged, np.array([1.0, 2.0, 3.0]), "z -= ... changes the caller's array; use z - ... instead"),
    ])


def check_split_indices(f):
    def expected(n, ratio, seed):
        perm = np.random.default_rng(seed).permutation(n)
        k = round(n * ratio)
        return perm[k:], perm[:k]

    def covers():
        tr, te = f(50, 0.2, 3)
        return (len(tr), len(te), sorted(np.concatenate([tr, te]).tolist()) == list(range(50)))

    _scenarios("split_indices", [
        ("follows the rule exactly", lambda: f(10, 0.3, 0), expected(10, 0.3, 0),
         "perm = np.random.default_rng(seed).permutation(n); test is the first round(n * ratio)"),
        ("sizes and coverage", covers, (40, 10, True), None),
        ("same seed, same split", lambda: _same(f(30, 0.2, 7), f(30, 0.2, 7)), True, None),
        ("ratio 0", lambda: len(f(5, 0.0, 1)[1]), 0, None),
        ("integer indexes", lambda: f(5, 0.4, 1)[0].dtype.kind, "i", None),
    ])


def _naive_dists(A, B):
    return np.array([[np.sqrt(((a - b) ** 2).sum()) for b in B] for a in A])


def check_pairwise_distances(f):
    A = _rng.normal(size=(6, 3))
    B = _rng.normal(size=(4, 3))

    def no_nan():
        P = _rng.normal(1000, 1, size=(50, 5))
        with np.errstate(invalid="ignore"):
            D = f(P, P)
        return (bool(np.isnan(D).any()), bool((D < 0).any()))

    _scenarios("pairwise_distances", [
        ("matches the direct formula", lambda: f(A, B), _naive_dists(A, B), None),
        ("shape (n, m)", lambda: f(A, B).shape, (6, 4), None),
        ("simple case", lambda: f(np.array([[0.0, 0.0]]), np.array([[3.0, 4.0], [0.0, 1.0]])), np.array([[5.0, 1.0]]), None),
        ("no NaN or negative values for identical points", no_nan, (False, False),
         "rounding can make a squared distance like -1e-9; clip at 0 before np.sqrt"),
    ])


def check_knn_predict(f):
    X_train = np.array([[0.0], [1.0], [2.0], [10.0], [11.0], [12.0]])
    y_train = np.array([0, 0, 0, 1, 1, 1])

    def against_naive():
        Xtr = _rng.normal(size=(60, 3))
        ytr = _rng.integers(0, 3, size=60)
        Xte = _rng.normal(size=(15, 3))
        D = _naive_dists(Xte, Xtr)
        expected = []
        for row in D:
            labels = ytr[np.argsort(row, kind="stable")[:5]]
            counts = np.bincount(labels, minlength=3)
            expected.append(int(np.argmax(counts)))
        return _same(f(Xtr, ytr, Xte, 5), np.array(expected))

    _scenarios("knn_predict", [
        ("simple", lambda: f(X_train, y_train, np.array([[1.5], [10.5]]), 3), np.array([0, 1]), None),
        ("k = 1", lambda: f(X_train, y_train, np.array([[5.0], [7.0]]), 1), np.array([0, 1]), None),
        ("a tie goes to the smaller label", lambda: f(np.array([[0.0], [2.0]]), np.array([1, 0]), np.array([[1.0]]), 2), np.array([0]),
         "count votes per class and take argmax, which picks the smallest label on a tie"),
        ("matches a direct implementation", against_naive, True,
         "count the votes for each class (np.add.at or np.bincount) and take argmax; don't average the labels"),
    ])


def check_confusion_matrix(f):
    _scenarios("confusion_matrix", [
        ("basic", lambda: f(np.array([0, 0, 1, 2, 1, 0]), np.array([0, 1, 1, 2, 0, 0]), 3),
         np.array([[2, 1, 0], [1, 1, 0], [0, 0, 1]]),
         "rows are true labels, columns are predictions. If some counts are too low, M[t, p] += 1 is ignoring repeated pairs: use np.add.at or np.bincount"),
        ("integer dtype", lambda: f(np.array([0]), np.array([0]), 2).dtype.kind, "i", None),
        ("classes that never appear", lambda: f(np.array([1]), np.array([1]), 4).shape, (4, 4), None),
        ("repeated pairs add up", lambda: f(np.array([1, 1, 1]), np.array([0, 0, 0]), 2), np.array([[0, 0], [3, 0]]),
         "M[t, p] += 1 with fancy indexing only adds 1 once for repeated pairs. Use np.add.at or np.bincount"),
        ("label out of range", lambda: f(np.array([0, 3]), np.array([0, 0]), 3), Raises(ValueError), None),
    ])


def check_moving_average(f):
    x = _rng.normal(size=200)
    naive = np.array([x[i:i + 7].mean() for i in range(len(x) - 6)])
    _scenarios("moving_average", [
        ("small example", lambda: f(np.array([1.0, 2.0, 3.0, 4.0, 5.0]), 2), np.array([1.5, 2.5, 3.5, 4.5]),
         "prepend a 0 to the cumulative sum so the first window works: c[w:] - c[:-w]"),
        ("window = length", lambda: f(np.array([2.0, 4.0]), 2), np.array([3.0]), None),
        ("matches the direct calculation", lambda: f(x, 7), naive,
         "prepend a 0 to the cumulative sum so the first window works: c[w:] - c[:-w]"),
        ("window 0", lambda: f(np.array([1.0, 2.0]), 0), Raises(ValueError), None),
        ("window too big", lambda: f(np.array([1.0, 2.0]), 3), Raises(ValueError), None),
    ])


def check_minibatches(f):
    X = np.arange(20).reshape(10, 2)
    y = np.arange(10)

    def in_order():
        return [(xb[:, 0].tolist(), yb.tolist()) for xb, yb in f(X, y, 4)]

    def shuffled_ok():
        batches = list(f(X, y, 3, shuffle=True, seed=0))
        rows = np.concatenate([yb for _, yb in batches])
        aligned = all((xb[:, 0] == 2 * yb).all() for xb, yb in batches)
        return (sorted(rows.tolist()) == list(range(10)), rows.tolist() != list(range(10)), aligned)

    def reproducible():
        a = [yb.tolist() for _, yb in f(X, y, 3, shuffle=True, seed=5)]
        b = [yb.tolist() for _, yb in f(X, y, 3, shuffle=True, seed=5)]
        return a == b

    _scenarios("minibatches", [
        ("is a generator", lambda: inspect.isgenerator(f(X, y, 2)), True, "use yield"),
        ("in order, last batch smaller", in_order,
         [([0, 2, 4, 6], [0, 1, 2, 3]), ([8, 10, 12, 14], [4, 5, 6, 7]), ([16, 18], [8, 9])], None),
        ("shuffled: every row once, X and y still matched", shuffled_ok, (True, True, True),
         "shuffle one array of indexes and use it for both X and y"),
        ("same seed, same order", reproducible, True, None),
        ("X and y of different lengths", lambda: next(f(X, y[:5], 2)), Raises(ValueError), None),
    ])


def check_ridge_fit(f):
    xs = np.linspace(0, 5, 20)
    X1 = xs[:, None]
    y1 = 2 * xs + 1

    def multi():
        X = _rng.normal(size=(100, 3))
        y = X @ np.array([1.5, -2.0, 0.5]) + 4.0
        w, b = f(X, y)
        return (w, b)

    def shrinks():
        X = _rng.normal(size=(50, 2))
        y = X @ np.array([3.0, -1.0]) + _rng.normal(0, 0.1, size=50)
        w0, _ = f(X, y, 0.0)
        w1, _ = f(X, y, 100.0)
        return bool(np.linalg.norm(w1) < np.linalg.norm(w0))

    def bias_not_penalised():
        X = _rng.normal(size=(200, 1))
        y = 0.0 * X[:, 0] + 50.0
        _, b = f(X, y, 1000.0)
        return b

    _scenarios("ridge_fit", [
        ("exact line, no regularization", lambda: f(X1, y1), (np.array([2.0]), 1.0), None),
        ("several features", multi, (np.array([1.5, -2.0, 0.5]), 4.0), None),
        ("l2 shrinks the weights", shrinks, True, None),
        ("the bias isn't regularized", bias_not_penalised, 50.0,
         "center X and y first, solve for w, then b = mean(y) - mean(X) @ w"),
    ])


def check_cross_entropy_with_grad(f):
    def numerical():
        logits = _rng.normal(size=(4, 3))
        y = np.array([0, 2, 1, 2])
        _, grad = f(logits, y)
        num = np.zeros_like(logits)
        h = 1e-6
        for i in range(4):
            for j in range(3):
                plus, minus = logits.copy(), logits.copy()
                plus[i, j] += h
                minus[i, j] -= h
                num[i, j] = (f(plus, y)[0] - f(minus, y)[0]) / (2 * h)
        return bool(np.allclose(grad, num, atol=1e-6))

    _scenarios("cross_entropy_with_grad", [
        ("uniform logits give log(k)", lambda: f(np.zeros((2, 4)), np.array([0, 3]))[0], float(np.log(4)), None),
        ("a confident correct prediction has a small loss", lambda: f(np.array([[10.0, 0.0]]), np.array([0]))[0] < 1e-3, True, None),
        ("large logits stay finite", lambda: np.isfinite(f(np.array([[1000.0, -1000.0]]), np.array([1]))[0]), True,
         "use log-softmax: z - max - log(sum(exp(z - max))), never log(softmax(z))"),
        ("gradient rows sum to 0", lambda: f(_rng.normal(size=(5, 3)), np.array([0, 1, 2, 0, 1]))[1].sum(axis=1), np.zeros(5), None),
        ("gradient matches a numerical estimate", numerical, True, "grad = (softmax(logits) - one_hot(y)) / n"),
    ])


def check_conv2d(f):
    def naive(img, ker):
        kh, kw = ker.shape
        H, W = img.shape
        out = np.zeros((H - kh + 1, W - kw + 1))
        for i in range(out.shape[0]):
            for j in range(out.shape[1]):
                out[i, j] = (img[i:i + kh, j:j + kw] * ker).sum()
        return out

    img = _rng.normal(size=(9, 12))
    ker = _rng.normal(size=(3, 4))
    edge = np.array([[0, 0, 1, 1]] * 4, dtype=float)
    _scenarios("conv2d", [
        ("matches the direct calculation", lambda: f(img, ker), naive(img, ker),
         "out[i, j] = (image[i:i+kh, j:j+kw] * kernel).sum(), with the kernel not flipped"),
        ("output shape", lambda: f(np.zeros((10, 8)), np.zeros((3, 3))).shape, (8, 6), None),
        ("vertical edge detector", lambda: f(edge, np.array([[-1.0, 1.0]])), np.array([[0.0, 1.0, 0.0]] * 4),
         "this is cross-correlation: don't flip the kernel"),
        ("kernel bigger than the image", lambda: f(np.zeros((2, 2)), np.zeros((3, 1))), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `standardize`

In [2]:
import numpy as np


def standardize(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std = np.where(std == 0, 1.0, std)
    return (X - mean) / std

In [3]:
check_standardize(standardize)

PASS  all 5 tests passed for standardize


`mean` and `std` have shape `(n_features,)`, and broadcasting lines them up with the last axis of `X`, so every row is shifted and scaled by its column's numbers.

`np.where(std == 0, 1.0, std)` swaps any zero standard deviation for 1. A constant column then becomes `(x - mean) / 1`, which is all zeros. Comparing `std == 0` exactly is fine here: a constant column gives a standard deviation of exactly 0.

`np.asarray(X, dtype=float)` converts integer input to float. Without it, an integer array would still work for this calculation, since `/` always produces floats, but converting up front avoids surprises in functions that use operations like `-=`.

`X - mean` creates a new array, so the caller's `X` is never touched. An in-place version (`X -= mean`) would change the caller's data, and would fail outright on an integer array, because NumPy can't store float results back into integer memory.

### Exercise 2: `one_hot`

In [4]:
import numpy as np


def one_hot(labels: np.ndarray, n_classes: int) -> np.ndarray:
    labels = np.asarray(labels)
    if labels.size and (labels.min() < 0 or labels.max() >= n_classes):
        raise ValueError(f"labels must be in [0, {n_classes}), got min {labels.min()}, max {labels.max()}")
    out = np.zeros((len(labels), n_classes))
    out[np.arange(len(labels)), labels] = 1.0
    return out

In [5]:
check_one_hot(one_hot)

PASS  all 5 tests passed for one_hot


`out[np.arange(n), labels] = 1.0` is a fancy-indexing pattern worth memorising. The two index arrays are paired up element by element: row 0 with `labels[0]`, row 1 with `labels[1]`, and so on, so exactly one cell per row is set. You'll see the same pattern in reverse in exercise 11, to pick out the predicted probability of the correct class for each sample.

A shorter alternative is `np.eye(n_classes)[labels]`: take the identity matrix and pick its rows.

The validation matters more than it looks. Negative indexes are legal in NumPy, so without the check, a label of `-1` would silently set the *last* column, and your model would train on wrong targets without any error. `labels.size and ...` skips the check for an empty array, where `min()` would raise an error.

### Exercise 3: `softmax`

In [6]:
import numpy as np


def softmax(z: np.ndarray) -> np.ndarray:
    shifted = z - z.max(axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)

In [7]:
check_softmax(softmax)

PASS  all 5 tests passed for softmax


This is the max-subtraction trick from the Module 2 spam filter, now for a whole batch at once. `np.exp(1000)` overflows to infinity, and infinity divided by infinity is NaN. Subtracting each row's maximum first doesn't change the answer, because the shift cancels out between the top and bottom of the fraction, but it makes the largest exponent exactly `exp(0) = 1`, so nothing can overflow.

`axis=-1` means "the last axis", which is why the same code handles shapes `(k,)` and `(n, k)`. `keepdims=True` keeps the reduced axis with size 1, so `z.max(...)` has shape `(n, 1)` and broadcasts back across each row. Without `keepdims`, a `(n,)` array would be lined up against the *columns* instead, giving wrong results or a shape error.

`z - ...` creates a new array. `z -= ...` would modify the caller's logits, which is the kind of bug that only shows up later, when the same logits are reused.

### Exercise 4: `split_indices`

In [8]:
import numpy as np


def split_indices(n: int, test_ratio: float, seed: int):
    perm = np.random.default_rng(seed).permutation(n)
    n_test = round(n * test_ratio)
    return perm[n_test:], perm[:n_test]

In [9]:
check_split_indices(split_indices)

PASS  all 5 tests passed for split_indices


Splitting *indexes* instead of the data itself is the safe pattern from the lesson. You apply the same index array to `X` and to `y`, so each row keeps its own label. Shuffling `X` and `y` separately is one of the classic bugs that produces a model with exactly chance-level accuracy and no error.

The function only needs `n`, not the data. That means the same split can be applied to several arrays that belong together, like features, labels and sample weights, and the indexes can be saved to a file so the exact same split can be recreated later.

The new generator API (`default_rng(seed)`) gives each call its own random stream. Calling it twice with the same seed gives the same split, no matter what other random numbers the program has drawn in between.

## Medium

### Exercise 5: `pairwise_distances`

In [10]:
import numpy as np


def pairwise_distances(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    sq = (A ** 2).sum(axis=1)[:, None] - 2 * A @ B.T + (B ** 2).sum(axis=1)[None, :]
    return np.sqrt(np.maximum(sq, 0.0))

In [11]:
check_pairwise_distances(pairwise_distances)

PASS  all 4 tests passed for pairwise_distances


The expansion turns the cross term for every pair into one matrix multiplication, `A @ B.T`, of shape `(n, m)`, plus two vectors of squared lengths broadcast across it. Memory use is just the `(n, m)` result.

**Why clip?** In exact maths, a squared distance can't be negative. In floating point, the three terms are each large and almost cancel out for nearby points, so the result can come out as something like `-3e-12`. `np.sqrt` of that is NaN, and one NaN distance can quietly break a k-NN prediction. `np.maximum(sq, 0.0)` fixes it. This is a standard precaution, and scikit-learn's `euclidean_distances` does the same.

**The broadcasting alternative** is exact and easy to read:

```python
diff = A[:, None, :] - B[None, :, :]       # (n, m, d)
return np.sqrt((diff ** 2).sum(axis=2))
```

but it builds an `(n, m, d)` array, which uses `d` times more memory. For 5,000 × 5,000 points with 512 features, that's over 100 GB. The expansion is the version that scales.

### Exercise 6: `knn_predict`

In [12]:
import numpy as np


def knn_predict(X_train, y_train, X_test, k: int) -> np.ndarray:
    D = pairwise_distances(X_test, X_train)                  # (n_test, n_train)
    nearest = np.argsort(D, axis=1, kind="stable")[:, :k]    # (n_test, k)
    labels = y_train[nearest]                                # (n_test, k)
    n_classes = int(y_train.max()) + 1
    votes = np.zeros((len(X_test), n_classes), dtype=int)
    np.add.at(votes, (np.arange(len(X_test))[:, None], labels), 1)
    return votes.argmax(axis=1)

In [13]:
check_knn_predict(knn_predict)

PASS  all 4 tests passed for knn_predict


Reading the shapes is the easiest way to follow it:

1. `D` is `(n_test, n_train)`: every distance.
2. `nearest` is `(n_test, k)`: the positions of the `k` closest training points for each test point.
3. `labels` is `(n_test, k)`: their labels, via fancy indexing with a 2D index array.
4. `votes` is `(n_test, n_classes)`: a count per class per test point.
5. `argmax` along `axis=1` gives one prediction per test point, and on a tie it returns the first maximum, which is the smallest label.

`np.add.at(votes, (rows, labels), 1)` adds 1 for every (test point, neighbour label) pair, and it correctly adds several times when the same pair appears several times. You'll meet the reason this needs `np.add.at` instead of `votes[rows, labels] += 1` in exercise 7.

`np.argsort` sorts each row fully. For large training sets, `np.argpartition(D, k, axis=1)[:, :k]` from the lesson is faster, since it only needs to find the `k` smallest, not put everything in order. We used `argsort` with `kind="stable"` here so that when two neighbours are at exactly the same distance, the earlier training point wins, which makes the result fully predictable.

### Exercise 7: `confusion_matrix`

In [14]:
import numpy as np


def confusion_matrix(y_true, y_pred, n_classes: int) -> np.ndarray:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    for arr in (y_true, y_pred):
        if arr.size and (arr.min() < 0 or arr.max() >= n_classes):
            raise ValueError(f"labels must be in [0, {n_classes})")
    flat = y_true * n_classes + y_pred
    return np.bincount(flat, minlength=n_classes ** 2).reshape(n_classes, n_classes)

In [15]:
check_confusion_matrix(confusion_matrix)

PASS  all 5 tests passed for confusion_matrix


**The bug in `M[y_true, y_pred] += 1`:** with fancy indexing, `+=` reads all the selected cells, adds 1 to each, and writes them back. If the same `(t, p)` pair appears three times, the same cell is read three times as 0, and written three times as 1. The final value is 1, not 3. No error, just undercounting. That's what the "repeated pairs add up" check tests. `np.add.at(M, (y_true, y_pred), 1)` is the fix that does accumulate repeats.

**The `bincount` trick** used in the solution is another fix, and a fast one. Each `(t, p)` pair is turned into a single number, `t * n_classes + p`, which is unique for every cell. `np.bincount` counts how often each number appears, and `reshape` turns the flat counts back into a square. `minlength` makes sure classes that never appear still get rows and columns.

This "combine several small integers into one index, count, reshape" idea is useful well beyond confusion matrices, for example to count co-occurrences, or to build 2D histograms.

### Exercise 8: `moving_average`

In [16]:
import numpy as np


def moving_average(x: np.ndarray, window: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if not 1 <= window <= len(x):
        raise ValueError(f"window must be between 1 and {len(x)}, got {window}")
    c = np.concatenate([[0.0], np.cumsum(x)])
    return (c[window:] - c[:-window]) / window

In [17]:
check_moving_average(moving_average)

PASS  all 5 tests passed for moving_average


`c[i]` is the sum of the first `i` values. The sum of any window from position `i` to `i + window` is then a single subtraction, `c[i + window] - c[i]`, and slicing does that for every window at once: `c[window:] - c[:-window]`.

The leading `0.0` is what makes the very first window work. Without it, `c[0]` would be `x[0]` rather than "the sum of nothing", and every window would be shifted by one, which is what the check's hint points at.

`c[:-window]` is safe here because `window` is at least 1. With a window of 0, `c[:-0]` would be `c[:0]`, an empty array, the same `-0` slicing trap from Modules 1 and 5. The validation rules that case out before it can happen.

For very long series of large numbers, the cumulative sum can build up rounding error. `np.convolve(x, np.ones(window) / window, mode="valid")` is an alternative that avoids it, at the cost of speed for big windows.

### Exercise 9: `minibatches`

In [18]:
import numpy as np


def minibatches(X, y, batch_size: int, shuffle: bool = False, seed=None):
    if len(X) != len(y):
        raise ValueError(f"X has {len(X)} rows but y has {len(y)}")
    idx = np.arange(len(X))
    if shuffle:
        idx = np.random.default_rng(seed).permutation(len(X))
    for start in range(0, len(X), batch_size):
        batch = idx[start:start + batch_size]
        yield X[batch], y[batch]

In [19]:
check_minibatches(minibatches)

PASS  all 5 tests passed for minibatches


One index array, shuffled once, applied to both `X` and `y`: the same safe pattern as `split_indices`. The `for` loop here runs once per *batch*, not once per sample, so it's cheap. The work inside each step is all array operations.

Because `X[batch]` uses fancy indexing, every batch is a **copy**. That's slightly slower than a view, but it means a training step can modify its batch (for example, data augmentation in place) without touching the original dataset.

Since this is a generator, the length check only runs when the first batch is requested, which is why the check uses `next(...)`. That's usually fine for a training loop, but if you'd rather fail at the moment of the call, split it into a normal function that validates and then returns an inner generator, as in Module 4's `batched` exercise.

You'll use exactly this function in the mini project to train a neural network.

## Hard

### Exercise 10: `ridge_fit`

In [20]:
import numpy as np


def ridge_fit(X: np.ndarray, y: np.ndarray, l2: float = 0.0):
    x_mean = X.mean(axis=0)
    y_mean = y.mean()
    Xc = X - x_mean
    yc = y - y_mean
    A = Xc.T @ Xc + l2 * np.eye(X.shape[1])
    w = np.linalg.solve(A, Xc.T @ yc)
    b = y_mean - x_mean @ w
    return w, float(b)

In [21]:
check_ridge_fit(ridge_fit)

PASS  all 4 tests passed for ridge_fit


**Why center?** After centering, the best-fitting line always passes through the origin, so the bias drops out of the equation entirely. We solve for the weights alone, then put the bias back at the end. If you instead added a column of ones to `X` and solved for everything together, the penalty would apply to the bias too, pulling it toward 0, which is what the "bias isn't regularized" check catches. A bias of 50 should stay 50 however strongly you penalise the weights.

**What the penalty does.** Adding `l2` to the diagonal makes large weights expensive, so the solution shrinks toward zero. That helps when features are highly correlated or there are more features than samples: plain least squares can then produce huge, unstable weights, while ridge stays well behaved. In fact, with `l2 > 0`, the matrix `A` can always be solved, even when `Xc.T @ Xc` alone can't.

**Why `solve`, not `inv`.** `np.linalg.solve(A, b)` finds `w` directly. `np.linalg.inv(A) @ b` computes the whole inverse matrix first, which is slower and loses more precision. It's a rule worth following whenever you see an inverse in a formula.

For small, dense datasets, this is essentially what scikit-learn's `Ridge` computes, in eight lines.

### Exercise 11: `cross_entropy_with_grad`

In [22]:
import numpy as np


def cross_entropy_with_grad(logits: np.ndarray, y: np.ndarray):
    n = logits.shape[0]
    shifted = logits - logits.max(axis=1, keepdims=True)
    log_sum_exp = np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    log_probs = shifted - log_sum_exp                     # log-softmax, computed stably
    loss = -log_probs[np.arange(n), y].mean()

    grad = np.exp(log_probs)                              # softmax
    grad[np.arange(n), y] -= 1.0                          # minus one-hot
    return float(loss), grad / n

In [23]:
check_cross_entropy_with_grad(cross_entropy_with_grad)

PASS  all 5 tests passed for cross_entropy_with_grad


**Stability.** Computing `np.log(softmax(z))` directly fails for large logits: the softmax of the losing class rounds to exactly 0, and `log(0)` is minus infinity. The solution computes the log-probabilities directly, as `shifted - log(sum(exp(shifted)))`, which never takes the log of something that has rounded to zero. This is the **log-sum-exp trick**, and it's why PyTorch's `CrossEntropyLoss` takes raw logits instead of probabilities.

**Picking the right entries.** `log_probs[np.arange(n), y]` is the paired fancy indexing from exercise 2: one entry per row, the log-probability of that row's correct class.

**The gradient.** `softmax - one_hot` has a nice meaning: for the correct class, the gradient is `p - 1` (negative, so gradient descent pushes that logit *up*), and for every wrong class it's `p` (positive, pushing those logits *down*), each in proportion to how wrong the model currently is. A confident, correct prediction gives a gradient close to zero, so the model stops changing things that already work. And since the probabilities in a row sum to 1 and the one-hot sums to 1, each row of the gradient sums to 0, which is one of the checks.

**Checking the gradient.** The check compares your gradient against a numerical estimate, by nudging each logit up and down a tiny amount and measuring the change in loss. This *gradient check* is the most important test for any hand-written gradient, the same idea as in the Module 3 autograd project. The mini project uses this function to train a neural network, and runs the same check on the whole network.

### Exercise 12: `conv2d`

In [24]:
import numpy as np


def conv2d(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    kh, kw = kernel.shape
    H, W = image.shape
    if kh > H or kw > W:
        raise ValueError(f"kernel {kernel.shape} is bigger than image {image.shape}")
    windows = np.lib.stride_tricks.sliding_window_view(image, (kh, kw))   # (H-kh+1, W-kw+1, kh, kw)
    return np.einsum("ijkl,kl->ij", windows, kernel)

In [25]:
check_conv2d(conv2d)

PASS  all 4 tests passed for conv2d


**`sliding_window_view`** returns a 4D array where `windows[i, j]` is the `(kh, kw)` patch of the image starting at `(i, j)`. The impressive part is that it copies nothing: it's a *view* that reads the same memory with cleverly chosen strides (how many bytes to step along each axis). A million overlapping patches cost no extra memory.

**`np.einsum("ijkl,kl->ij", ...)`** is a compact way to write "multiply each patch by the kernel and sum over the patch's two axes". The letters name the axes: `k` and `l` appear in both inputs and not in the output, so they're multiplied and summed away. It's worth learning to read `einsum`, because attention in transformers is often written with it. `(windows * kernel).sum(axis=(2, 3))` gives the same result, but builds the full product array first.

**The other approach** loops over the kernel instead of the image:

```python
out = np.zeros((H - kh + 1, W - kw + 1))
for a in range(kh):
    for b in range(kw):
        out += kernel[a, b] * image[a:a + out.shape[0], b:b + out.shape[1]]
```

A 3×3 kernel means only 9 iterations, each one a whole-array operation, however big the image is.

**What it's for.** The check's edge detector, `[[-1, 1]]`, responds only where brightness jumps from left to right. In a CNN, the kernels aren't designed by hand like this. They're learned by gradient descent, and the first layer of a trained image model typically ends up with kernels that look like exactly these kinds of edge and texture detectors.

## What's next

The mini project in `04_mini_project.ipynb` builds a neural network in NumPy that learns to read handwritten digits, using the same softmax, cross-entropy and mini-batch code as these exercises.